# 02 — Feature Engineering

Loads all raw CSVs, builds the full Phase 2 feature set via `src/features.py`, and saves `data/processed/laps_features.csv`.

**Run order:**
1. Load & clean race laps
2. Build driver skill score (2023+2024 only)
3. Build all features
4. Inspect output
5. Save

In [69]:
import sys, os
sys.path.append("..")

import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")

import importlib
import src.features as feat
importlib.reload(feat)
from src.features import (
    build_features, build_driver_skill,
    parse_laptimes, add_relative_pace,
    FEATURE_COLUMNS, TARGET_COLUMN
)

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
RAW_DIR  = os.path.join(BASE_DIR, "data", "raw")
PROC_DIR = os.path.join(BASE_DIR, "data", "processed")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
print("Imports OK")

Imports OK


## 1. Load raw CSVs

In [70]:
YEARS = [2023, 2024, 2025]

def load_csv(name, year):
    path = os.path.join(RAW_DIR, f"{name}_{year}.csv")
    if os.path.exists(path):
        return pd.read_csv(path)
    print(f"  missing: {name}_{year}.csv")
    return pd.DataFrame()

race_parts  = [load_csv("race",        y) for y in YEARS]
fp2_parts   = [load_csv("fp2_profile", y) for y in YEARS]
fp3_parts   = [load_csv("fp3_profile", y) for y in YEARS]
fp1_parts   = [load_csv("fp1_profile", y) for y in YEARS]
quali_parts = [load_csv("quali",       y) for y in YEARS]

race_df  = pd.concat([p for p in race_parts  if not p.empty], ignore_index=True)
fp2_df   = pd.concat([p for p in fp2_parts   if not p.empty], ignore_index=True)
fp3_df   = pd.concat([p for p in fp3_parts   if not p.empty], ignore_index=True)
fp1_df   = pd.concat([p for p in fp1_parts   if not p.empty], ignore_index=True)
quali_df = pd.concat([p for p in quali_parts if not p.empty], ignore_index=True)

print(f"race_df  : {len(race_df):,} rows")
print(f"fp2_df   : {len(fp2_df):,} rows")
print(f"fp3_df   : {len(fp3_df):,} rows")
print(f"fp1_df   : {len(fp1_df):,} rows")
print(f"quali_df : {len(quali_df):,} rows")

race_df  : 77,720 rows
fp2_df   : 1,309 rows
fp3_df   : 1,356 rows
fp1_df   : 1,302 rows
quali_df : 1,394 rows


## 2. Clean race laps

Keep only clean, representative flying laps:
- Remove pit in/out laps
- Remove laps with no recorded time
- Remove unrealistically slow laps (safety car, red flag, lap 1 chaos)
- Keep only dry-weather compounds (SOFT / MEDIUM / HARD)
- Keep only `IsAccurate == True` laps

In [71]:
def parse_td(val):
    try:
        return pd.to_timedelta(val).total_seconds()
    except:
        return None

race_df["LapTimeSeconds"] = race_df["LapTime"].apply(parse_td)

total_before = len(race_df)

clean = race_df[
    race_df["LapTimeSeconds"].notna() &
    race_df["PitInTime"].isna() &
    race_df["PitOutTime"].isna() &
    race_df["IsAccurate"].fillna(False) &
    race_df["Compound"].isin(["SOFT", "MEDIUM", "HARD"]) &
    (race_df["LapNumber"] > 1)
].copy()

# Remove safety-car / VSC laps: anything >40% slower than the event median
event_med = clean.groupby(["Year", "EventName"])["LapTimeSeconds"].transform("median")
clean = clean[clean["LapTimeSeconds"] <= event_med * 1.40]

print(f"Before cleaning : {total_before:,} laps")
print(f"After cleaning  : {len(clean):,} laps")
print(f"Removed         : {total_before - len(clean):,} ({(total_before - len(clean))/total_before*100:.1f}%)")
print(f"\nRounds per year : {clean.groupby('Year')['RoundNumber'].nunique().to_dict()}")

Before cleaning : 77,720 laps
After cleaning  : 63,915 laps
Removed         : 13,805 (17.8%)

Rounds per year : {2023: 22, 2024: 23, 2025: 24}


In [72]:
# Missing values check on key columns
key_cols = ["LapTimeSeconds", "TyreLife", "Compound", "Team",
            "TrackTemp", "Rainfall", "CircuitType", "RoundNumber"]
missing = clean[key_cols].isnull().sum()
print("Missing values:")
print(missing[missing > 0] if missing.any() else "None")

Missing values:
TyreLife    19
dtype: int64


## 3. Build driver skill score

Built from 2023+2024 data only — never includes current season to prevent leakage.
Saved to `data/processed/driver_skill.csv`.

In [73]:
# build_driver_skill needs RelativePace — compute it on the clean race data first
race_for_skill = parse_laptimes(clean.copy())
race_for_skill = add_relative_pace(race_for_skill)

skill_df = build_driver_skill(race_for_skill, quali_df)

print("\nTop 10 drivers by skill score:")
print(skill_df.sort_values("DriverSkillScore", ascending=False).head(10).to_string(index=False))

  Saved driver_skill.csv (25 drivers)

Top 10 drivers by skill score:
Driver  DriverSkillScore
   VER             0.896
   BOT             0.775
   ALO             0.760
   ALB             0.755
   TSU             0.699
   NOR             0.672
   HUL             0.646
   LEC             0.624
   GAS             0.616
   RUS             0.610


## 4. Build full feature set

In [74]:
df_features = build_features(
    race_df  = clean,
    fp2_df   = fp2_df,
    fp3_df   = fp3_df,
    fp1_df   = fp1_df,
    quali_df = quali_df,
    skill_df = skill_df,
)

Building features...
  ✓ Basic features done
  ✓ TeamPerformanceIndex done
  ✓ FP2 features merged
  ✓ Quali features merged
  ✓ FP3 features merged
  ✓ FP1 features merged
  ✓ DriverSkillScore merged
  ✓ RollingSeasonForm done
  ✓ TrackCharacterScore done
  ✓ WeekendPaceScore done
✅ Feature engineering complete. Shape: (70891, 74)


## 5. Inspect output

In [75]:
print(f"Shape: {df_features.shape}")
present   = [c for c in FEATURE_COLUMNS if c in df_features.columns]
missing_f = [c for c in FEATURE_COLUMNS if c not in df_features.columns]
print(f"Features present : {len(present)}/{len(FEATURE_COLUMNS)}")
if missing_f:
    print(f"MISSING          : {missing_f}")

Shape: (70891, 74)
Features present : 23/23


In [76]:
# NaN rates per feature
nan_rates = df_features[FEATURE_COLUMNS].isnull().mean().sort_values(ascending=False)
print("NaN rates:")
has_nan = nan_rates[nan_rates > 0]
print(has_nan.apply(lambda x: f"{x*100:.1f}%") if len(has_nan) > 0 else "None")

NaN rates:
FP2DegRate          15.5%
FP2LongRunPace      15.1%
DriverSkillScore     4.8%
DegradationRate      1.1%
QualiTeammateGap     0.7%
GapToPole            0.5%
QualiPosition        0.3%
TyreLife             0.0%
dtype: object


In [77]:
df_features[FEATURE_COLUMNS + [TARGET_COLUMN]].describe().T

,count,mean,std,min,25%,50%,75%,max
TyreLife,70872.000,15.871,10.813,1.000,8.000,14.000,22.000,78.000
CompoundEncoded,70891.000,2.404,0.679,1.000,2.000,3.000,3.000,3.000
Compound_SOFT,70891.000,0.110,0.313,0.000,0.000,0.000,0.000,1.000
Compound_MEDIUM,70891.000,0.376,0.484,0.000,0.000,0.000,1.000,1.000
Compound_HARD,70891.000,0.514,0.500,0.000,0.000,1.000,1.000,1.000
DegradationRate,70122.000,0.030,0.045,-0.287,0.006,0.042,0.057,0.187
LapNumber,70891.000,32.010,17.930,2.000,17.000,32.000,46.000,78.000
FuelLoad,70891.000,0.475,0.283,0.000,0.227,0.474,0.719,0.972
TrackEvolution,70891.000,0.525,0.283,0.028,0.281,0.526,0.773,1.000
TrackEncoded,70891.000,11.918,6.891,0.000,6.000,12.000,18.000,23.000


In [78]:
# Feature correlation with lap time
numeric_feats = [c for c in FEATURE_COLUMNS
                 if df_features[c].dtype in [float, int, np.float64, np.int64]]

corr = (
    df_features[numeric_feats + [TARGET_COLUMN]]
    .corr()[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
    .sort_values()
)

fig = px.bar(
    x=corr.values, y=corr.index,
    orientation="h",
    title="Feature correlation with LapTimeSeconds",
    labels={"x": "Pearson r", "y": "Feature"},
    color=corr.values,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    height=650,
)
fig.update_layout(showlegend=False)
fig.show()

In [79]:
# Tyre degradation sanity check — lap time should rise with tyre life
sample = df_features[df_features["Year"] == 2024].copy()
sample = sample[sample["TyreLife"] <= 40]

fig = px.scatter(
    sample.sample(min(5000, len(sample)), random_state=42),
    x="TyreLife", y="LapTimeSeconds",
    color="Compound",
    color_discrete_map={"SOFT": "red", "MEDIUM": "yellow", "HARD": "white"},
    opacity=0.4,
    trendline="ols",
    title="Lap time vs Tyre life (2024 sample)",
    height=450,
)
fig.show()

In [80]:
# Driver skill ranking
fig = px.bar(
    skill_df.sort_values("DriverSkillScore", ascending=True),
    x="DriverSkillScore", y="Driver",
    orientation="h",
    title="Driver Skill Score (built from 2023+2024)",
    height=700,
)
fig.show()

## 6. Train / test split check

- **Train:** 2023, 2024, 2025 rounds 1–18  
- **Test:** 2025 rounds 19+

In [81]:
train_mask = (
    df_features["Year"].isin([2023, 2024]) |
    ((df_features["Year"] == 2025) & (df_features["RoundNumber"] <= 18))
)
test_mask = (df_features["Year"] == 2025) & (df_features["RoundNumber"] > 18)

print(f"Train laps : {train_mask.sum():,}")
print(f"Test laps  : {test_mask.sum():,}")
print(f"Test rounds: {sorted(df_features[test_mask]['RoundNumber'].unique())}")

Train laps : 64,146
Test laps  : 6,745
Test rounds: [np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]


## 7. Save

In [82]:
os.makedirs(PROC_DIR, exist_ok=True)
out_path = os.path.join(PROC_DIR, "laps_features.csv")
df_features.to_csv(out_path, index=False)

print(f"Saved  : {out_path}")
print(f"Shape  : {df_features.shape}")
print(f"Columns: {list(df_features.columns)}")

Saved  : c:\yashas\f1-strategy-optimizer\data\processed\laps_features.csv
Shape  : (70891, 74)
Columns: ['Time_x', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate', 'Year', 'RoundNumber', 'EventName', 'SessionType', 'CircuitType', 'Time_y', 'AirTemp', 'TrackTemp', 'Humidity', 'WindSpeed', 'Rainfall', 'LapTimeSeconds', 'Sector1Seconds', 'Sector2Seconds', 'Sector3Seconds', 'CompoundEncoded', 'Compound_SOFT', 'Compound_MEDIUM', 'Compound_HARD', 'Compound_INTERMEDIATE', 'Compound_WET', 'TotalLaps', 'TrackEvolution', 'FuelLoad', 'SessionMedianLapTime', 'RelativePace', 'TrackTempNorm', 'TempCompoundInteraction', 'Degradation